# Child Well-being - POSet creation
**University: University of Milano-Bicocca**  
**Master's Degree: Data Science (A.Y. 2025/2026)**  
**Course: Data Science Lab**  

---  
Use the Python Porting of R package `poseticDataAnalysis` by Avellone, De Capitani, Fattore.  
Added some implementations to work with Polars dataframes and handle `null` values ​​as uncertainty intervals in the hyperlattice.
**reference:**  
Fattore M., De Capitani L., Avellone A., Suardi A. (2024).  
*A fuzzy posetic toolbox for multi-criteria evaluation on ordinal data systems.*  
Annals of Operations Research. [doi:10.1007/s10479-024-06352-3](https://link.springer.com/article/10.1007/s10479-024-06352-3?utm_source=researchgate.net&utm_medium=article)

In [ ]:
import sys; sys.path.insert(0, '..')
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
import poset as P

---
## 0. Parameters summary -> `poset_from_polars`

| Parameter | Default | Description |
|-----------|---------|-------------|
| `col1` | `None` | First column to generate the ID |
| `col2` | `None` | Second column to generate the ID |
| `id_col` | `None` | Pre-existing ID (alternative to col1+col2) |
| `indicator_cols` | `None` | Indicator columns; if None -> all int |
| `higher_is_better` | `True` | High value = better position |
| `dominance_mode` | `'certain_or_possible'` | Type of dominance to calculate (certain, possible, certain_or_possible)  |
| `value_range` | `None` | Theoretical range `(min, max)` for null intervals |
| `max_null_frac` | `1.0` | Max null fraction to include a unit |
| `unit_sep` | `'_'` | ID separator |

**Dictionary key output returned:**
- `poset_certain` / `poset_possible` — the two POSets to use with all the library functions
- `intervals` — `{unit: {'lo': array, 'hi': array}}` — the region of each unit
- `confidence` — `{(a,b): float}` — reliability of each dominance pair
- `null_mask` — `{unit: bool array}` — where are the nulls

## 1. Import datasets

In [ ]:
# Import dataset
ind_2018 = pl.read_parquet('../data/040_indicators_2018.parquet')
ind_2015 = pl.read_parquet('../data/040_indicators_2015.parquet')
exp_2018 = pl.read_parquet('../data/040_public_expenditure_2018.parquet') 
exp_2015 = pl.read_parquet('../data/040_public_expenditure_2015.parquet')
datasets = [ind_2018, ind_2015, exp_2018, exp_2015]
print('Data loaded successfully')

---

## 2. Data extration and POSet creation

Since the `POSet(elements, dom)` constructor accepts `elements` as a list of labels and `dom` as a list of pairs `(a, b)` with `a ≤ b`, I need to extract the two lists from the datasets.  
The remaining `null` values are not missing data to be imputed; instead, they are treated as structural uncertainty: the unit does not occupy a single point in the hyperlattice, but rather a region, namely all points compatible with the observed values.  

The function uses the following normalization:
- every unit **a** with `null` on indicator **k** is compatible with any value along that dimension
- therefore, **a** represents an interval in the hyperlattice: `[a_lower, a_upper]`, where `a_lower` has the `null` values replaced with the minimum possible values and `a_upper` with the maximum
- Dominance then becomes interval dominance:
   - `a ≤ certain b` if `a_upper ≤ b_lower` (certainly below)
   - `a ≤ possible b` if `a_lower ≤ b_upper` (possibly below)
   - otherwise, they are incomparable


In [ ]:
# Extract elemments and dominance from dtatasets

results = {}

for df in datasets:
    df_name = [k for k,v in globals().items() if v is df][0]
    print(f'Processing dataset [{df_name}]')
    result = P.poset_from_polars(
        df,
        col1='REF_AREA',      # area first element for ID
        col2='TIME_PERIOD',   # year second element for ID
        higher_is_better=True,
        dominance_mode='certain_or_possible',
    )
    results[df_name] = result

    

In [ ]:
import pickle
import os

path = f"..{os.sep}data{os.sep}050_posets.pkl"

with open(path, "wb") as f:
    pickle.dump(results, f, protocol=pickle.HIGHEST_PROTOCOL)

### 2.0 Selct the dataset to explore

In [ ]:
dfsel = exp_2018
result_sel = results['exp_2018'] # select the results for the dataset you want to explore

### 2.1 Ranges visualization

In [ ]:
# plot intervals for POSet elements

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import hsv_to_rgb

units    = result_sel['elements'] # select the results for the dataset you want to plot
intervals = result_sel['intervals']
cols = dfsel.columns
n_ind    = len(cols) - 2  # excluded REF_AREA and TIME_PERIOD
sel_indicators = cols[2:]  # assuming indicators start from the 3rd column

fig, ax = plt.subplots(figsize=(16, 8))

hues = np.linspace(0, 1, len(units), endpoint=False)
colors = hsv_to_rgb([
    (h, 0.75, 0.9) for h in hues
])


for ui, (unit, col) in enumerate(zip(units, colors)):
    lo = intervals[unit]['lo']
    hi = intervals[unit]['hi']
    nm = result_sel['null_mask'][unit]
    offset = (ui - len(units)/2) * 0.12

    for j in range(n_ind):
        x = j + offset
        if not nm[j]:  # value -> point
            ax.plot(x, lo[j], 'o', color=col, ms=8, zorder=3)
        else:          # null -> interval
            ax.plot([x, x], [lo[j], hi[j]], '-', color=col, lw=4,
                    alpha=0.6, solid_capstyle='round')
            ax.plot(x, lo[j], 'v', color=col, ms=7, zorder=3)
            ax.plot(x, hi[j], '^', color=col, ms=7, zorder=3)

ax.set_xticks(range(n_ind))
ax.set_xticklabels(sel_indicators, rotation=30, ha='right')
ax.set_ylabel('Indicator value')
ax.set_title('Intervals for POSet elements (● = observed, ↕ = uncertainty from null)')
ax.set_ylim(0.5, 4.5)
ax.grid(axis='y', alpha=0.3)

patches = [mpatches.Patch(color=colors[i], label=units[i]) for i in range(len(units))]

legend = ax.legend(
    handles=patches,
    loc='upper center',
    bbox_to_anchor=(0.5, -0.22),
    ncol=min(6, len(units)),
    frameon=False,
    fontsize=9
)

plt.subplots_adjust(bottom=0.30)
plt.show()

### 2.2 Summary diagnostic
`interval_summary` returns the volume of the region of each unit:  
- for a country with no nulls it is 1 (point)  
- for a country with 3 nulls on range [1,5] it is 5^3 = 125 — a direct measure of how big its uncertainty zone is.

In [ ]:
summary = P.interval_summary(result_sel)
pl.Config.set_tbl_rows(-1)       
pl.Config.set_tbl_cols(-1)        
display(summary)
print()
print('Interpretation interval_volume:')
print('  → volume = product of the widths of each dimension (indicator)')
print('  → volume = 1   if no null values (point)')
print('  → volume > 1   if there are null values (region in the hypercube)')

In [ ]:
# Plot volume of uncertainty per unit
units_s = summary['unit'].to_list()
volumes  = summary['interval_volume'].to_list()
n_nulls  = summary['n_null'].to_list()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10))

# Volume di indeterminazione
bar_colors = ['#e74c3c' if v > 1 else '#2ecc71' for v in volumes]
ax1.bar(units_s, volumes, color=bar_colors, edgecolor='white')
ax1.axhline(1, color='black', lw=1.5, ls='--', label='Point (vol=1)')
ax1.set_ylabel('Interval volume (log scale)')
ax1.set_yscale('log')
ax1.set_title('Positional uncertainty in the poset')
ax1.legend()
ax1.tick_params(axis='x', rotation=20)

# Null count
ax2.bar(units_s, n_nulls, color='#3498db', edgecolor='white')
ax2.set_ylabel('N. null indicators')
ax2.set_title('Null per unit')
ax2.tick_params(axis='x', rotation=20)

plt.suptitle('Uncertainty diagnosis — selected indicators', fontsize=12)
plt.tight_layout()
plt.show()

## 3. POSet comparison - certain vs. possible

In [ ]:
def plot_hasse(poset, title, ax):
    covers = P.CoverRelation(poset)
    G = nx.DiGraph()
    G.add_nodes_from(poset.elements)
    G.add_edges_from(covers)
    try:
        layout = nx.nx_agraph.graphviz_layout(G, prog='dot')
    except:
        layout = nx.spring_layout(G, seed=42)
    minimals = set(P.POSetMinimals(poset))
    maximals = set(P.POSetMaximals(poset))
    colors = ['#2ecc71' if n in maximals else
              '#e74c3c' if n in minimals else
              '#3498db' for n in poset.elements]
    nx.draw(G, layout, ax=ax, with_labels=True, node_color=colors,
            node_size=1800, font_color='white', font_size=10, font_weight='bold',
            edge_color='#555', arrows=True, arrowsize=18)
    ax.set_title(title, fontsize=12)


fig, axes = plt.subplots(2, 1, figsize=(16, 16))
plot_hasse(result_sel['poset_certain'],  'POSet CERTAIN\n(a ≤ b in every scenario)', axes[0])
plot_hasse(result_sel['poset_possible'], 'POSet POSSIBLE\n(a ≤ b in at least one scenario)', axes[1])

patches = [
    mpatches.Patch(color='#2ecc71', label='Maximin'),
    mpatches.Patch(color='#e74c3c', label='Minimum'),
    mpatches.Patch(color='#3498db', label='Intermediate'),
]
fig.legend(handles=patches, loc='lower center', ncol=3, fontsize=10)
plt.suptitle('Confronto POSet certo vs possibile', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

### 3.1 Confidence as a measure of dominance reliability

In [ ]:
print('Intervalli:')
for uid in result_sel['elements']:
    lo = result_sel['intervals'][uid]['lo']
    hi = result_sel['intervals'][uid]['hi']
    nm = result_sel['null_mask'][uid]
    inds = result_sel['indicator_cols']
    print(f'  {uid}:', end=' ')
    for j, ind in enumerate(inds):
        if nm[j]:
            print(f'{ind}=[{lo[j]:.0f},{hi[j]:.0f}]', end=' ')
        else:
            print(f'{ind}={lo[j]:.0f}', end=' ')
    print()

In [ ]:
print('Certain dominance:', result_sel['dom_certain'])
print('Possible dominance:', result_sel['dom_possible'])
print()
print('Confidence (indicators with no null on either unit):')
for pair, c in sorted(result_sel['confidence'].items()):
    kind = 'CERTAIN' if pair in set(result_sel['dom_certain']) else 'Only Possible'
    print(f'  {pair[0]} ≤ {pair[1]}  →  {c:.0%}  [{kind}]')